Precision, Recall, F1-score
Clase 1 = impago

Precision: De los clientes que el modelo predice como impago, cuántos realmente incumplen.

Recall: De todos los clientes que realmente incumplen, cuántos el modelo predice correctamente.

F1-score: Media armónica entre precision y recall, útil para balancear ambos.


Tu objetivo es detectar impagos (clase 1). Por tanto:

Recall de clase 1 es la métrica más importante
→ quieres minimizar falsos negativos (clientes que incumplen pero tu modelo dice que no).

Precision importa menos que recall si estás dispuesto a aceptar algunos falsos positivos (alertas de riesgo innecesarias).

F1-score de clase 1 te da un balance, útil para comparar modelos.

ROC-AUC es buena métrica general de ranking de riesgo.

In [9]:
# =====================================================
# IMPORTS
# =====================================================
import numpy as np
import pandas as pd
import os

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestCentroid
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score,
    recall_score,
    precision_score,   # <--- FALTABA ESTO
    make_scorer,
    silhouette_score,
    precision_recall_curve
)

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.neighbors import NearestCentroid

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    StackingClassifier,
    AdaBoostClassifier
)
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import time

# =====================================================
# 1. CARGAR DATOS
# =====================================================
def cargar_y_preparar_datos(ruta_archivo):
    df = pd.read_excel(ruta_archivo)
    # Filtrar solo vivienda y copiar para evitar warnings
    df_viv = df[df['Proposito'].astype(str)
                .str.contains('Vivienda', case=False, na=False)].copy()
    # Label
    df_viv['Impago_Label'] = df_viv['Impago'].map({0:0, 1:1})
    return df_viv

# Ajusta esta ruta si es necesario
ruta_real = os.path.join('..', 'Datos', 'Limpios', 'información_préstamos_limpio.xlsx')

if os.path.exists(ruta_real):
    df = cargar_y_preparar_datos(ruta_real)
else:
    print(f"⚠️ ATENCIÓN: No se encuentra el archivo en {ruta_real}")
    # Crea un df dummy si no encuentra el archivo para que no rompa al copiar/pegar
    # (Borra esto en tu código real si ya tienes el archivo)
    df = pd.DataFrame() 

# =====================================================
# 2. DEFINIR X e y
# =====================================================
if not df.empty:
    target_col = "Impago_Label"
    columnas_a_eliminar = ["ID", "Impago", "Prima", "Proposito"]

    y = df[target_col]
    X = df.drop(columns=[target_col])
    X = X.drop(columns=[col for col in columnas_a_eliminar if col in X.columns])

    # Eliminar alta cardinalidad
    high_card_cols = [col for col in X.columns if X[col].nunique() > 50]
    X = X.drop(columns=high_card_cols)

    # One-hot encoding
    cat_cols = X.select_dtypes(include="object").columns
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

    X = X.astype("float32")

    # =====================================================
    # 3. TRAIN / TEST SPLIT
    # =====================================================
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.25,
        stratify=y,
        random_state=42
    )

    # =====================================================
    # 4. CLUSTERING: TORNEO K-MEANS vs AGLOMERATIVO
    # =====================================================
    print("--- Iniciando Optimización de Clustering ---")

    # 1. Escalado
    scaler_cluster = StandardScaler()
    X_train_cluster = scaler_cluster.fit_transform(X_train)
    X_test_cluster = scaler_cluster.transform(X_test)

    # --- PASO 1: ENCONTRAR EL NÚMERO DE CLUSTERS (K) ÓPTIMO ---
    # Probamos de 2 a 5 clusters y nos quedamos con el que tenga mejor Silueta
    print("🔎 Buscando el número óptimo de clusters (k)...")
    best_k = 3  # Valor por defecto
    best_k_score = -1

    for k in [2, 3, 4, 5]:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X_train_cluster)
        score = silhouette_score(X_train_cluster, labels)
        print(f"   k={k} -> Silhouette: {score:.4f}")
        
        if score > best_k_score:
            best_k_score = score
            best_k = k

    print(f"✅ Número óptimo seleccionado: k={best_k}")

    # --- PASO 2: TORNEO CON EL K GANADOR (KMeans vs Aglomerativo) ---
    print(f"\n--- Iniciando Torneo (usando k={best_k}) ---")

    scores = {}
    labels_storage = {}

    # Función auxiliar
    def asignar_clusters(model, X_train_scaled, X_test_scaled):
        labels_train = model.fit_predict(X_train_scaled)
        if hasattr(model, "predict"):
            labels_test = model.predict(X_test_scaled)
        else:
            centroid_clf = NearestCentroid()
            centroid_clf.fit(X_train_scaled, labels_train)
            labels_test = centroid_clf.predict(X_test_scaled)
        return labels_train, labels_test

    # Opción A: KMeans
    kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
    k_labels_train, k_labels_test = asignar_clusters(kmeans, X_train_cluster, X_test_cluster)
    scores["KMeans"] = silhouette_score(X_train_cluster, k_labels_train)
    labels_storage["KMeans"] = (k_labels_train, k_labels_test)
    print(f"Silhouette KMeans: {scores['KMeans']:.4f}")

    # Opción B: Aglomerativo
    agg = AgglomerativeClustering(n_clusters=best_k)
    a_labels_train, a_labels_test = asignar_clusters(agg, X_train_cluster, X_test_cluster)
    scores["Agglomerative"] = silhouette_score(X_train_cluster, a_labels_train)
    labels_storage["Agglomerative"] = (a_labels_train, a_labels_test)
    print(f"Silhouette Agglomerative: {scores['Agglomerative']:.4f}")

    # --- SELECCIÓN FINAL ---
    best_model_name = max(scores, key=scores.get)
    print(f"\n🏆 GANADOR DEL TORNEO: {best_model_name} con k={best_k}")

    # Aplicar al dataset
    final_labels_train, final_labels_test = labels_storage[best_model_name]

    # One-Hot Encoding
    train_dummies = pd.get_dummies(final_labels_train, prefix='Cluster_Group')
    test_dummies = pd.get_dummies(final_labels_test, prefix='Cluster_Group')

    # Alinear columnas
    test_dummies = test_dummies.reindex(columns=train_dummies.columns, fill_value=0)

    # Unir
    train_dummies.index = X_train.index
    test_dummies.index = X_test.index
    X_train = pd.concat([X_train, train_dummies], axis=1)
    X_test = pd.concat([X_test, test_dummies], axis=1)

    print(f"✅ Variables de cluster añadidas. Nuevas columnas: {list(train_dummies.columns)}")
    # =====================================================
    # 5. FUNCIÓN ENTRENAMIENTO PRIORITIZANDO RECALL
    # =====================================================
    # =====================================================
# 5. FUNCIÓN ENTRENAMIENTO (CON TIEMPOS Y GAP)
# =====================================================
def entrenar_modelo(
    nombre_modelo,
    modelo,
    param_grid,
    X_train, X_test,
    y_train, y_test,
    usar_smote=False,
    usar_pca=False,
    threshold=None 
):
    
    steps = [("scaler", StandardScaler())]

    if usar_smote:
        steps.append(("smote", SMOTE(random_state=42)))
    if usar_pca:
        steps.append(("pca", PCA(n_components=0.95, random_state=42)))

    steps.append(("model", modelo))
    pipe = ImbPipeline(steps)

    param_grid_pipeline = {f"model__{k}": v for k,v in param_grid.items()}
    recall_scorer = make_scorer(recall_score, pos_label=1)

    # --- 1. MEDIR TIEMPO DE ENTRENAMIENTO ---
    start_train = time.time()
    
    grid = GridSearchCV(pipe, param_grid_pipeline, cv=3, scoring=recall_scorer, n_jobs=-1)
    grid.fit(X_train, y_train)
    
    end_train = time.time()
    train_time = end_train - start_train  # Tiempo en segundos

    # --- 2. CÁLCULO DE THRESHOLD DINÁMICO ---
    # Medimos tiempo de predicción también
    start_pred = time.time()
    y_proba_test = grid.best_estimator_.predict_proba(X_test)[:, 1]
    
    if threshold is None:
        precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba_test)
        f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
        best_idx = np.argmax(f1_scores)
        best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    else:
        best_threshold = threshold
        
    y_pred_test = (y_proba_test >= best_threshold).astype(int)
    end_pred = time.time()
    prediction_time = end_pred - start_pred # Tiempo en segundos

    # --- 3. CÁLCULO DE METRICAS EN TRAIN (Para ver Overfitting) ---
    # Usamos el MISMO threshold que en test para ser justos
    y_proba_train = grid.best_estimator_.predict_proba(X_train)[:, 1]
    y_pred_train = (y_proba_train >= best_threshold).astype(int)
    
    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc = accuracy_score(y_test, y_pred_test)
    
    # GAP: Si es muy alto positivo (>10-15%), hay Overfitting
    gap = (train_acc - test_acc) * 100 

    # --- 4. OTRAS MÉTRICAS TEST ---
    roc = roc_auc_score(y_test, y_proba_test)
    recall1 = recall_score(y_test, y_pred_test, pos_label=1)
    precision1 = precision_score(y_test, y_pred_test, pos_label=1, zero_division=0)

    print("\n", "="*60)
    print(f"{nombre_modelo} | SMOTE={usar_smote} | PCA={usar_pca} | THRESH={best_threshold:.4f}")
    print(f" Tiempo Train: {train_time:.2f}s | Tiempo Pred: {prediction_time:.4f}s")
    print(f" Acc Train: {train_acc:.4f} | Acc Test: {test_acc:.4f} | GAP: {gap:.2f}%")
    print(" ROC-AUC:", round(roc,4))
    print(" Recall (Impago):", round(recall1,4))

    return {
        "Modelo": nombre_modelo,
        "SMOTE": usar_smote,
        "PCA": usar_pca,
        "Threshold": best_threshold,
        "Train_Time_Sec": train_time,     
        "Pred_Time_Sec": prediction_time,  
        "Train_Accuracy": train_acc,      
        "Test_Accuracy": test_acc,         
        "Overfitting_Gap_Pct": gap,         
        "ROC_AUC": roc,
        "Recall_1": recall1,
        "Precision_1": precision1
    }

    # 6. DEFINIR MODELOS
    modelos = {
        "LogReg": (LogisticRegression(max_iter=1000, class_weight="balanced"), {"C":[0.01,0.1,1]}),
        "RandomForest": (RandomForestClassifier(random_state=42, class_weight="balanced"), {"n_estimators":[100,200]}),
        "DecisionTree": (DecisionTreeClassifier(random_state=42, class_weight="balanced"), {"max_depth":[None,5,10]}),
        "AdaBoost": (AdaBoostClassifier(random_state=42), {"n_estimators":[50,100]}),
        "XGBoost": (XGBClassifier(eval_metric="logloss", random_state=42, use_label_encoder=False),
                    {"n_estimators":[100], "max_depth":[3,6]})
    }

    estimadores_base = [
        ("rf", RandomForestClassifier(n_estimators=100, random_state=42)),
        ("dt", DecisionTreeClassifier(random_state=42)),
        ("nb", GaussianNB())
    ]

    stacking = StackingClassifier(
        estimators=estimadores_base,
        final_estimator=LogisticRegression()
    )

    modelos["Stacking"] = (stacking, {"final_estimator__C":[0.1,1]})

    # 7. EJECUCIÓN PARA TODAS LAS COMBINACIONES
    combinaciones = [
        (False, False), # 1. Nada
        (True, False),  # 2. Solo SMOTE
        (False, True),  # 3. Solo PCA
        (True, True)    # 4. SMOTE + PCA
    ]

    resultados_finales = []

    for nombre, (modelo, grid) in modelos.items():
        for smote_flag, pca_flag in combinaciones:
            
            res = entrenar_modelo(
                nombre, modelo, grid,
                X_train, X_test,
                y_train, y_test,
                usar_smote=smote_flag,
                usar_pca=pca_flag,
                threshold=None 
            )
            
            resultados_finales.append(res)

    df_resultados = pd.DataFrame(resultados_finales)

--- Iniciando Optimización de Clustering ---
🔎 Buscando el número óptimo de clusters (k)...
   k=2 -> Silhouette: 0.1469
   k=3 -> Silhouette: 0.1329
   k=4 -> Silhouette: 0.1505
   k=5 -> Silhouette: 0.1456
✅ Número óptimo seleccionado: k=4

--- Iniciando Torneo (usando k=4) ---
Silhouette KMeans: 0.1505
Silhouette Agglomerative: 0.1801

🏆 GANADOR DEL TORNEO: Agglomerative con k=4
✅ Variables de cluster añadidas. Nuevas columnas: ['Cluster_Group_0', 'Cluster_Group_1', 'Cluster_Group_2', 'Cluster_Group_3']


In [8]:
print("\n=========== RESULTADOS FINALES ===========")  
print(df_resultados.sort_values("Recall_1", ascending=False))     # Ordenamos por Recall porque es tu prioridad en Riesgo


=========== RESULTADOS FINALES ===========
          Modelo  SMOTE    PCA  Threshold  Accuracy   ROC_AUC  Recall_1  \
23      Stacking   True   True   0.045725  0.196997  0.540920  0.964744   
6   RandomForest  False   True   0.003038  0.214940  0.526220  0.910256   
21      Stacking   True  False   0.051610  0.252655  0.535305  0.897436   
18       XGBoost  False   True   0.032725  0.410106  0.569111  0.772436   
3         LogReg   True   True   0.446143  0.475650  0.607111  0.762821   
22      Stacking  False   True   0.106804  0.454412  0.586607  0.753205   
20      Stacking  False  False   0.098544  0.458806  0.594682  0.750000   
0         LogReg  False  False   0.469607  0.493592  0.612326  0.750000   
1         LogReg   True  False   0.456011  0.491761  0.608628  0.740385   
11  DecisionTree   True   True   0.492632  0.452215  0.576539  0.730769   
2         LogReg  False   True   0.477243  0.507506  0.607898  0.721154   
10  DecisionTree  False   True   0.513233  0.495423  0.5

🏆 El Verdadero Ganador: AdaBoost o Regresión Logística
Tienes que buscar el equilibrio: un Recall alto (>70%) pero con un AUC decente (>0.60).

Mis recomendaciones son:

OPCIÓN A (La más equilibrada): AdaBoost | Sin SMOTE | Sin PCA (Fila 12)

Recall: 74.36% (Detecta a 3 de cada 4 morosos).

ROC-AUC: 0.614 (El más alto de la tabla).

Por qué elegirlo: Es el que mejor "entiende" la diferencia entre pagar e impagar (mayor AUC) manteniendo un Recall muy alto.

OPCIÓN B (La más técnica): LogReg | SMOTE=True | PCA=True (Fila 3)

Recall: 75.96% (Un pelín mejor capturando morosos).

ROC-AUC: 0.607 (Muy estable).

Precision: 0.148 (Baja, como todos).

Por qué elegirlo: La Regresión Logística es el estándar en banca porque es explicable. Usar PCA y SMOTE demuestra dominio técnico.